```{contents}
```

## Training Failure Modes, Convergence Diagnostics, and Optimization Dynamics

### Why These Topics Matter

Modern deep learning success depends less on model architecture and more on **how training behaves**.
Understanding *failure modes*, *diagnostics*, and *optimization dynamics* allows practitioners to:

* Detect problems early
* Choose appropriate hyperparameters
* Stabilize and accelerate convergence
* Improve generalization

---

### Training Failure Modes

#### Underfitting

Model is too simple or insufficiently trained.

**Symptoms**

* High training loss
* High validation loss

**Causes**

* Insufficient capacity
* Too much regularization
* Inadequate training time

**Fixes**

* Increase model depth/width
* Reduce regularization
* Train longer

---

#### Overfitting

Model memorizes training data.

**Symptoms**

* Training loss ↓
* Validation loss ↑

**Causes**

* Excessive capacity
* Small dataset
* Poor regularization

**Fixes**

* Data augmentation
* Dropout, weight decay
* Early stopping

---

#### Vanishing / Exploding Gradients

Common in deep or recurrent networks.

**Symptoms**

* Gradients → 0 (learning stalls)
* Gradients → ∞ (loss diverges)

**Fixes**

* Proper initialization (Xavier, He)
* Batch normalization
* Residual connections
* Gradient clipping

---

#### Optimization Instability

Loss oscillates or diverges.

**Causes**

* Learning rate too high
* Poor normalization
* Bad initialization

**Fix**

* Lower learning rate
* Use Adam/AdamW
* Apply batch norm

---

### Convergence Diagnostics

#### Key Signals to Monitor

| Metric          | Purpose               |
| --------------- | --------------------- |
| Training loss   | Optimization progress |
| Validation loss | Generalization        |
| Accuracy / F1   | Task performance      |
| Gradient norms  | Stability             |
| Learning rate   | Step magnitude        |

---

#### Interpreting Training Curves

| Pattern                 | Interpretation      |
| ----------------------- | ------------------- |
| Loss decreases smoothly | Healthy convergence |
| Loss oscillates         | LR too high         |
| Loss plateaus early     | LR too low          |
| Validation diverges     | Overfitting         |
| Both high               | Underfitting        |

---

### Optimization Dynamics

Deep learning optimization is **non-convex**, but exhibits special structure:

* Many flat low-loss regions
* Wide minima correlate with better generalization
* SGD noise helps escape sharp minima

#### Common Optimizers

| Optimizer | Dynamics                            |
| --------- | ----------------------------------- |
| SGD       | Stable, slower, good generalization |
| Adam      | Fast convergence, adaptive          |
| AdamW     | Better regularization               |
| RMSprop   | Good for RNNs                       |

---

### PyTorch Demonstration: Detecting Failures



```python
for epoch in range(30):
    model.train()
    train_loss = 0
    for X, y in train_loader:
        optimizer.zero_grad()
        loss = criterion(model(X), y)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    model.eval()
    val_loss = 0
    with torch.no_grad():
        for X, y in val_loader:
            val_loss += criterion(model(X), y).item()

    print(epoch, train_loss, val_loss)
```



#### Example Diagnosis

```text
Epoch 5  Train: 0.20  Val: 0.45
Epoch 6  Train: 0.15  Val: 0.60   ← overfitting begins
```

Apply early stopping:

```python
if val_loss > best_val:
    patience += 1
    if patience > 3:
        break
```

---

### Monitoring Gradient Health

```python
total_norm = 0
for p in model.parameters():
    param_norm = p.grad.data.norm(2)
    total_norm += param_norm.item() ** 2
total_norm = total_norm ** 0.5
```

Large values → exploding gradients
Tiny values → vanishing gradients

---

### Stabilization Techniques

| Technique                | Effect                    |
| ------------------------ | ------------------------- |
| Batch Normalization      | Smooths optimization      |
| Gradient Clipping        | Prevents explosions       |
| Learning rate scheduling | Faster convergence        |
| Warmup                   | Prevents early divergence |
| Residual connections     | Improves gradient flow    |

---

### Key Insight

> Training behavior is an emergent property of architecture, data, optimization, and initialization.

Controlling this behavior is the difference between **failed training and state-of-the-art performance**.

